In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading Dataset ##

In [27]:
df=pd.read_csv("/Users/deepanshus/StockMarketPredictor/Project/data/raw/Banks/AUFI Historical Data.csv")
df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,13-03-2026,884.35,900.00,905.85,881.90,2.56M,-1.98%
1,12-03-2026,902.20,911.10,912.75,899.10,1.83M,-1.76%
2,11-03-2026,918.40,937.30,947.50,914.40,1.68M,-2.14%
3,10-03-2026,938.50,941.20,951.30,929.10,1.97M,0.74%
4,09-03-2026,931.65,943.40,948.30,913.60,2.56M,-3.38%


## Preprocessing ##

In [28]:
df =df.rename(columns={
    "Price":"close",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Vol.":"volume",
    "Change %":"change_percent"

})
df['date']=pd.to_datetime(df['Date'],format="%d-%m-%Y")

df = df.sort_values("date")
df = df.reset_index(drop=True)

In [29]:
# # chaning volume

df.head()
df.isna().sum()

Date              0
close             0
open              0
high              0
low               0
volume            1
change_percent    0
date              0
dtype: int64

In [30]:
df[df.isnull().any(axis=1)]

,Date,close,open,high,low,volume,change_percent,date
825,18-05-2024,624.30,624.30,624.30,624.30,NaN,-0.01%,2024-05-18


In [31]:
df.drop(825,inplace=True)

In [32]:
df.isna().sum()

Date              0
close             0
open              0
high              0
low               0
volume            0
change_percent    0
date              0
dtype: int64

In [33]:
def convert_volume(x):
    
    if isinstance(x, str):

        if "M" in x:
            return float(x.replace("M","")) * 1_000_000

        elif "K" in x:
            return float(x.replace("K","")) * 1_000

        else:
            return float(x)

    return x

df["volume"]=df["volume"].apply(convert_volume)

In [34]:
df['change_percent']=df['change_percent'].str.replace('%',"").astype(float)


In [35]:
# cols = ["open","high","low","close"]

# for c in cols:
#     df[c] = pd.to_numeric(df[c])]

cols = ["open", "high", "low", "close"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

## Calculating Indicators and making Dataset

In [39]:
# creating functions for calculating features
def sort_dataset(df):

    df = df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    return df
def calculate_returns(df):

    df["return"] = df["close"].pct_change()

    return df
def calculate_volume_change(df):

    df["volume_change"] = df["volume"].pct_change()

    return df
def calculate_volatility(df):

    df["volatility"] = df["return"].rolling(10).std()

    return df
def calculate_sma(df):

    df["sma10"] = df["close"].rolling(10).mean()

    df["sma50"] = df["close"].rolling(50).mean()

    df["sma_ratio"] = df["sma10"] / df["sma50"]

    return df
def calculate_macd(df):

    ema12 = df["close"].ewm(span=12, adjust=False).mean()

    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["macd"] = ema12 - ema26

    return df
def calculate_rsi(df):

    delta = df["close"].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()

    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["rsi"] = 100 - (100 / (1 + rs))

    return df
def create_target(df):

    df["target"] = df["close"].shift(-1) / df["close"] - 1

    return df
def clean_dataset(df):

    df = df.dropna()

    return df

In [41]:
# calling functions
df = sort_dataset(df)

df = calculate_returns(df)

df = calculate_volume_change(df)

df = calculate_volatility(df)

df = calculate_sma(df)

df = calculate_macd(df)

df = calculate_rsi(df)

df = create_target(df)

df = clean_dataset(df)

indicator_df = df[[
    "rsi",
    "macd",
    "sma_ratio",
    "volume_change",
    "volatility",
    "target"
]]
indicator_df.to_csv("indicator_dataset.csv", index=False)